In [6]:
import logging
import os

from huggingface_hub import hf_hub_download

# local .py file
from scPRINT import (
    extract_model_weights,
    load_scprint,
    populate_lamin_db,
    SCPRINT_DEFS
)
from utils import (
    save_results,
    load_results,
    RESULTS_DEFS,
    compute_attention_from_weights
)

logger = logging.getLogger(__name__)


In [2]:
# Configuration
DATA_DIR = "data"
OUTPUT_DIR = "output"

MODEL_PATH = os.path.join(DATA_DIR, "scPRINT")
os.makedirs(MODEL_PATH, exist_ok=True)


In [3]:
populate_lamin_db()

# 1. Download and load model
logger.info("Downloading/loading model if needed ...")
model_checkpoint_file = hf_hub_download(
    repo_id="jkobject/scPRINT", 
    filename="v2-medium.ckpt", 
    cache_dir=MODEL_PATH
)

# Load model and data
logger.info("Loading scPRINT model")
model, gene_annotations, model_metadata = load_scprint(model_checkpoint_file)

# Extract weights
logger.info("Extracting model weights")
weights_dict = extract_model_weights(model)

# Save using standardized format
logger.info(f"Saving weights to {OUTPUT_DIR}")
save_results(weights_dict, gene_annotations, model_metadata, OUTPUT_DIR, SCPRINT_DEFS.MODEL_NAME)

Mon Oct  6 13:54:06 2025 INFO Lamin database already configured
Mon Oct  6 13:54:06 2025 INFO Downloading/loading model if needed ...
Mon Oct  6 13:54:06 2025 INFO Loading scPRINT model
Mon Oct  6 13:54:06 2025 INFO Loading scPRINT model
Mon Oct  6 13:54:12 2025 INFO Loading gene annotations


RuntimeError caught: scPrint is not attached to a `Trainer`.


Mon Oct  6 13:54:13 2025 INFO Formatting model metadata
Mon Oct  6 13:54:13 2025 INFO Extracting model weights
Mon Oct  6 13:54:13 2025 INFO Saving weights to output
Mon Oct  6 13:54:13 2025 INFO Saving weights to output/scPRINT_weights.npz and metadata to output/scPRINT_metadata.json
Mon Oct  6 13:54:13 2025 INFO Successfully validated weights, gene metadata and model metadata
Mon Oct  6 13:54:13 2025 INFO Saving weights to output/scPRINT_weights.npz
Mon Oct  6 13:54:13 2025 INFO Saving metadata to output/scPRINT_metadata.json
Mon Oct  6 13:54:13 2025 INFO Successfully saved all results


In [ ]:
weights_dict, gene_annotations, model_metadata = load_results(OUTPUT_DIR, SCPRINT_DEFS.MODEL_NAME)

GENES_OF_INTEREST = gene_annotations[RESULTS_DEFS.VOCAB_NAME].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in model_metadata[RESULTS_DEFS.ORDERED_VOCABULARY]]

# Compute attention on demand
layer_attn = compute_attention_from_weights(
    weights_dict[RESULTS_DEFS.GENE_EMBEDDING][GENE_MASK,:],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_5'][RESULTS_DEFS.W_Q],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_5'][RESULTS_DEFS.W_K]
)

Mon Oct  6 13:54:57 2025 INFO Loading weights from output/scPRINT_weights.npz and metadata from output/scPRINT_metadata.json
Mon Oct  6 13:54:57 2025 INFO Loading weights from output/scPRINT_weights.npz
Mon Oct  6 13:54:57 2025 INFO Loading metadata from output/scPRINT_metadata.json
Mon Oct  6 13:54:57 2025 INFO Successfully loaded and validated all results
